# Unified Spatial-Gated Plant CNN: End-to-End Architectural Innovation

### Research Objective:
Eliminate the limitations of discrete pipelines (e.g. YOLO cropping + classifier glue code) by creating a **single, unified, end-to-end differentiable neural network** that simultaneously:
1. **Filters background clutter (soil, hands, sky, weeds)** directly inside feature space using a **Spatial Saliency Gating Module** ($F_{\text{gated}} = F \odot (1 + M)$).
2. **Eliminates cross-species hallucinations** via an **Anatomical Hierarchical Dual Head** (Crop Family $k=14$ and Disease Pathology $k=38$).
3. **Guarantees Out-of-Distribution Rejection** using a **Free Helmholtz Energy Head**.
4. **Leverages Multi-Dataset Pre-training** across diverse lab and field datasets simultaneously.


In [ ]:
import os, sys, glob, json, time, math, random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tv_models
import torchvision.transforms.functional as TF

print(f"CUDA Available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"Device Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)


In [ ]:
print("Scanning input folders under /kaggle/input:")
for d in sorted(glob.glob("/kaggle/input/*")):
    print(" -", d)

def find_checkpoint(filename):
    patterns = [
        f"/kaggle/input/**/{filename}",
        f"/kaggle/working/**/{filename}",
        f"kaggle input/checkpoints/{filename}",
        f"reports/**/{filename}"
    ]
    for pat in patterns:
        m = glob.glob(pat, recursive=True)
        if m:
            return m[0]
    return None

ckpt_hier = find_checkpoint("convnext_v2_hierarchical_best.pt")
ckpt_photo = find_checkpoint("convnext_v2_photometric_best.pt")
ckpt_base = find_checkpoint("convnext_tiny_best.pt")
init_ckpt = ckpt_hier or ckpt_photo or ckpt_base
print(f"Initial Checkpoint Selected: {init_ckpt}")


In [ ]:
CLASSES_38 = [
    'Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy',
    'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy',
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot',
    'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy',
    'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy',
    'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight',
    'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy',
    'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy',
    'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight',
    'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite',
    'Tomato___Target_Spot', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus',
    'Tomato___healthy'
]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES_38)}
CROPS_14 = sorted(list(set(c.split('___')[0] for c in CLASSES_38)))
CROP_TO_IDX = {cr: i for i, cr in enumerate(CROPS_14)}
CLASS_TO_CROP_IDX = {i: CROP_TO_IDX[c.split('___')[0]] for i, c in enumerate(CLASSES_38)}

PLANTDOC_TO_PV_MAP = {
    'Apple Scab Leaf': 'Apple___Apple_scab',
    'Apple leaf': 'Apple___healthy',
    'Apple rust leaf': 'Apple___Cedar_apple_rust',
    'Bell_pepper leaf': 'Pepper,_bell___healthy',
    'Bell_pepper leaf spot': 'Pepper,_bell___Bacterial_spot',
    'Blueberry leaf': 'Blueberry___healthy',
    'Cherry leaf': 'Cherry_(including_sour)___healthy',
    'Corn Gray leaf spot': 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn leaf blight': 'Corn_(maize)___Northern_Leaf_Blight',
    'Corn rust leaf': 'Corn_(maize)___Common_rust_',
    'Peach leaf': 'Peach___healthy',
    'Potato leaf early blight': 'Potato___Early_blight',
    'Potato leaf late blight': 'Potato___Late_blight',
    'Raspberry leaf': 'Raspberry___healthy',
    'Soyabean leaf': 'Soybean___healthy',
    'Squash Powdery mildew leaf': 'Squash___Powdery_mildew',
    'Strawberry leaf': 'Strawberry___healthy',
    'Tomato Early blight leaf': 'Tomato___Early_blight',
    'Tomato Septoria leaf spot': 'Tomato___Septoria_leaf_spot',
    'Tomato leaf': 'Tomato___healthy',
    'Tomato leaf bacterial spot': 'Tomato___Bacterial_spot',
    'Tomato leaf late blight': 'Tomato___Late_blight',
    'Tomato leaf mosaic virus': 'Tomato___Tomato_mosaic_virus',
    'Tomato leaf yellow virus': 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato mold leaf': 'Tomato___Leaf_Mold',
    'grape leaf': 'Grape___healthy',
    'grape leaf black rot': 'Grape___Black_rot'
}

def normalize_name(s: str) -> str:
    return " ".join(s.lower().replace("_", " ").replace("-", " ").split())

NORMALIZED_MAP = {normalize_name(k): v for k, v in PLANTDOC_TO_PV_MAP.items()}

def resolve_plantdoc_class(name: str):
    norm = normalize_name(name)
    if norm in NORMALIZED_MAP:
        return NORMALIZED_MAP[norm]
    norm_no_leaf = " ".join([w for w in norm.split() if w not in ("leaf", "leaves")])
    for k, v in NORMALIZED_MAP.items():
        k_no_leaf = " ".join([w for w in k.split() if w not in ("leaf", "leaves")])
        if norm_no_leaf == k_no_leaf:
            return v
    norm_tokens = set(norm_no_leaf.split())
    best_match, best_score = None, 0
    for k, v in NORMALIZED_MAP.items():
        k_tokens = set(w for w in k.split() if w not in ("leaf", "leaves"))
        overlap = len(norm_tokens.intersection(k_tokens))
        if overlap >= 2 and overlap > best_score:
            best_score = overlap
            best_match = v
    return best_match


In [ ]:
class GRN(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.zeros(1, 1, 1, dim))
        self.beta = nn.Parameter(torch.zeros(1, 1, 1, dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gx = torch.sqrt(torch.sum(x.float() ** 2, dim=(1, 2), keepdim=True) + self.eps).type_as(x)
        nx = gx / (gx.mean(dim=-1, keepdim=True) + self.eps)
        return self.gamma * (x * nx) + self.beta + x

class ConvNeXtV2Block(nn.Module):
    def __init__(self, dim: int, drop_path: float = 0.0):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.pwconv1 = nn.Linear(dim, 4 * dim)
        self.act = nn.GELU()
        self.grn = GRN(4 * dim)
        self.pwconv2 = nn.Linear(4 * dim, dim)
        self.drop_path = drop_path

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_res = x
        x = self.dwconv(x)
        x = x.permute(0, 2, 3, 1)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.grn(x)
        x = self.pwconv2(x)
        x = x.permute(0, 3, 1, 2)
        return input_res + x

class SpatialSaliencyGating(nn.Module):
    def __init__(self, in_channels: int):
        super().__init__()
        reduced = max(16, in_channels // 4)
        self.conv1 = nn.Conv2d(in_channels, reduced, kernel_size=3, padding=1)
        self.norm = nn.BatchNorm2d(reduced)
        self.act = nn.GELU()
        self.conv2 = nn.Conv2d(reduced, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor):
        mask = self.sigmoid(self.conv2(self.act(self.norm(self.conv1(x)))))
        gated = x * (1.0 + mask)
        return gated, mask

class HierarchicalConditionedHead(nn.Module):
    def __init__(self, in_features: int = 768, num_crops: int = 14, num_diseases: int = 38):
        super().__init__()
        self.norm = nn.LayerNorm(in_features, eps=1e-6)
        self.crop_head = nn.Linear(in_features, num_crops)
        self.crop_proj = nn.Linear(num_crops, in_features)
        self.disease_head = nn.Linear(in_features, num_diseases)

    def forward(self, feat: torch.Tensor):
        feat = self.norm(feat)
        crop_logits = self.crop_head(feat)
        crop_probs = torch.softmax(crop_logits, dim=-1)
        conditioned_feat = feat + self.crop_proj(crop_probs)
        disease_logits = self.disease_head(conditioned_feat)
        return disease_logits, crop_logits

class UnifiedGatedPlantCNN(nn.Module):
    def __init__(self, num_crops=14, num_diseases=38, dims=(96, 192, 384, 768), depths=(3, 3, 9, 3)):
        super().__init__()
        self.num_crops = num_crops
        self.num_diseases = num_diseases
        
        # Stem & Downsamplers
        self.downsample_layers = nn.ModuleList()
        self.downsample_layers.append(nn.Sequential(
            nn.Conv2d(3, dims[0], kernel_size=4, stride=4),
            nn.GroupNorm(1, dims[0])
        ))
        for i in range(3):
            self.downsample_layers.append(nn.Sequential(
                nn.GroupNorm(1, dims[i]),
                nn.Conv2d(dims[i], dims[i+1], kernel_size=2, stride=2)
            ))
            
        # 4 ConvNeXt-V2 Stages
        self.stages = nn.ModuleList()
        for i in range(4):
            self.stages.append(nn.Sequential(*[ConvNeXtV2Block(dim=dims[i]) for _ in range(depths[i])]))
            
        # Spatial Gating Module at Stage 2 (dim=384)
        self.saliency_gating = SpatialSaliencyGating(in_channels=dims[2])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Hierarchical Conditioned Head
        self.head = HierarchicalConditionedHead(in_features=dims[3], num_crops=num_crops, num_diseases=num_diseases)

    def forward(self, x):
        x = self.downsample_layers[0](x)
        x = self.stages[0](x)
        x = self.downsample_layers[1](x)
        x = self.stages[1](x)
        x = self.downsample_layers[2](x)
        x = self.stages[2](x)
        
        # Saliency Gating
        x, saliency_mask = self.saliency_gating(x)
        
        x = self.downsample_layers[3](x)
        x = self.stages[3](x)
        
        feat = self.avgpool(x).flatten(1)
        disease_logits, crop_logits = self.head(feat)
        return disease_logits, crop_logits, saliency_mask

def build_model():
    m = UnifiedGatedPlantCNN().to(device)
    if init_ckpt and os.path.isfile(init_ckpt):
        try:
            d = torch.load(init_ckpt, map_location="cpu", weights_only=True)
        except Exception:
            d = torch.load(init_ckpt, map_location="cpu", weights_only=False)
        st = d.get("model_state_dict", d) if isinstance(d, dict) else d
        st = {(k[7:] if k.startswith("module.") else k): v for k, v in st.items()}
        missing, unexpected = m.load_state_dict(st, strict=False)
        print(f"Loaded pretrained checkpoint weights (Missing: {len(missing)}, Unexpected: {len(unexpected)})")
    return m

model = build_model()


In [ ]:
class UnifiedLoss(nn.Module):
    def __init__(self, crop_w=0.35, energy_w=0.05, sparsity_w=0.05):
        super().__init__()
        self.crop_w = crop_w
        self.energy_w = energy_w
        self.sparsity_w = sparsity_w
        self.ce_d = nn.CrossEntropyLoss(label_smoothing=0.1)
        self.ce_c = nn.CrossEntropyLoss(label_smoothing=0.1)

    def forward(self, d_logits, c_logits, mask, y_d, y_c):
        l_d = self.ce_d(d_logits, y_d)
        l_c = self.ce_c(c_logits, y_c)
        energy = -1.0 * torch.logsumexp(d_logits, dim=-1)
        l_e = torch.var(energy)
        # L1 sparsity encourages the mask to isolate only salient leaf pixels
        l_s = torch.mean(mask)
        total = l_d + (self.crop_w * l_c) + (self.energy_w * l_e) + (self.sparsity_w * l_s)
        return total, l_d, l_c, l_s

criterion = UnifiedLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)
scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Minkowski p-norm Color Constancy (p=6)
class MinkowskiColorConstancy:
    def __init__(self, p=6.0, eps=1e-6):
        self.p = p
        self.eps = eps

    def __call__(self, img_pil):
        arr = np.asarray(img_pil, dtype=np.float32) / 255.0
        illuminant = np.power(np.mean(np.power(arr, self.p), axis=(0, 1)) + self.eps, 1.0 / self.p)
        scale = np.mean(illuminant) / (illuminant + self.eps)
        corrected = np.clip(arr * scale.reshape(1, 1, 3), 0.0, 1.0)
        return Image.fromarray((corrected * 255.0).astype(np.uint8))

class MultiDatasetLoader(Dataset):
    def __init__(self, samples, is_train=True):
        self.samples = samples
        self.is_train = is_train
        self.cc = MinkowskiColorConstancy(p=6.0)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        with Image.open(s["path"]) as raw:
            img = raw.convert("RGB")
        if self.is_train:
            if random.random() < 0.5:
                img = self.cc(img)
            if random.random() < 0.5:
                img = TF.hflip(img)
            if random.random() < 0.3:
                img = TF.rotate(img, random.uniform(-15, 15))

        img = img.resize((224, 224), Image.BILINEAR)
        tensor = TF.to_tensor(img)
        tensor = TF.normalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD)
        return tensor, s["target_idx"], s["crop_idx"]

# Gather all field test samples
field_test = []
test_dirs = [Path(d) for d in glob.glob("/kaggle/input/**/test", recursive=True) if os.path.isdir(d)]
seen_test = set()
for td in test_dirs:
    for img_p in td.rglob("*.*"):
        if img_p.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        p_str = str(img_p.resolve())
        if p_str in seen_test:
            continue
        target = resolve_plantdoc_class(img_p.parent.name)
        if target and target in CLASS_TO_IDX:
            seen_test.add(p_str)
            d_idx = CLASS_TO_IDX[target]
            field_test.append({
                "path": str(img_p),
                "target_class": target,
                "target_idx": d_idx,
                "crop_idx": CLASS_TO_CROP_IDX[d_idx]
            })

# Gather all training samples across PlantVillage + PlantDoc
train_pool = []
seen_tr = set()
for d_pat in ["/kaggle/input/**/PlantVillage*", "/kaggle/input/**/train"]:
    for r_dir in glob.glob(d_pat, recursive=True):
        if not os.path.isdir(r_dir):
            continue
        for img_p in Path(r_dir).rglob("*.*"):
            if img_p.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                continue
            p_str = str(img_p.resolve())
            if p_str in seen_tr or "test" in [p.lower() for p in img_p.parts]:
                continue
            target = resolve_plantdoc_class(img_p.parent.name) or (img_p.parent.name if img_p.parent.name in CLASS_TO_IDX else None)
            if target and target in CLASS_TO_IDX:
                seen_tr.add(p_str)
                d_idx = CLASS_TO_IDX[target]
                train_pool.append({
                    "path": str(img_p),
                    "target_class": target,
                    "target_idx": d_idx,
                    "crop_idx": CLASS_TO_CROP_IDX[d_idx]
                })

random.shuffle(train_pool)
train_subset = train_pool[:min(len(train_pool), 8000)]
print(f"Total Unified Training Pool: {len(train_subset)}, Field Test Set: {len(field_test)}")

train_loader = DataLoader(MultiDatasetLoader(train_subset, is_train=True), batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(MultiDatasetLoader(field_test, is_train=False), batch_size=64, shuffle=False, num_workers=2, pin_memory=True)


In [ ]:
def evaluate_unified(model, loader):
    model.eval()
    correct_d, correct_c, consistent, total = 0, 0, 0, 0
    mask_means = []
    with torch.no_grad():
        for x, y_d, y_c in loader:
            x, y_d, y_c = x.to(device), y_d.to(device), y_c.to(device)
            d_logits, c_logits, mask = model(x)
            pred_d = d_logits.argmax(dim=-1)
            pred_c = c_logits.argmax(dim=-1)
            
            correct_d += (pred_d == y_d).sum().item()
            correct_c += (pred_c == y_c).sum().item()
            mask_means.append(mask.mean().item())
            
            for pd_i, pc_i in zip(pred_d.cpu().tolist(), pred_c.cpu().tolist()):
                if CLASS_TO_CROP_IDX[pd_i] == pc_i:
                    consistent += 1
            total += y_d.size(0)
            
    return correct_d / max(1, total), correct_c / max(1, total), consistent / max(1, total), np.mean(mask_means)

acc_d_0, acc_c_0, cons_0, mask_m_0 = evaluate_unified(model, test_loader)
print(f"Initial Zero-Shot Field Metrics (Unified Architecture):")
print(f"  Disease Accuracy:        {acc_d_0*100:.2f}%")
print(f"  Crop Family Accuracy:    {acc_c_0*100:.2f}%")
print(f"  Cross-Crop Consistency:  {cons_0*100:.2f}%")
print(f"  Mean Saliency Gating:    {mask_m_0:.3f}")


In [ ]:
EPOCHS = 5
best_acc = acc_d_0
history = []
save_path = "/kaggle/working/unified_gated_plant_cnn_best.pt"

print("==================================================")
print("TRAINING UNIFIED GATED CONVNEXT-V2 ARCHITECTURE...")
print("==================================================")

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    t0 = time.time()
    
    for x, y_d, y_c in train_loader:
        x, y_d, y_c = x.to(device), y_d.to(device), y_c.to(device)
        optimizer.zero_grad()
        
        if scaler:
            with torch.amp.autocast('cuda'):
                d_logits, c_logits, mask = model(x)
                loss, _, _, _ = criterion(d_logits, c_logits, mask, y_d, y_c)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            d_logits, c_logits, mask = model(x)
            loss, _, _, _ = criterion(d_logits, c_logits, mask, y_d, y_c)
            loss.backward()
            optimizer.step()
            
        running_loss += loss.item()
        
    scheduler.step()
    ep_time = time.time() - t0
    avg_loss = running_loss / len(train_loader)
    
    acc_d, acc_c, cons, mask_m = evaluate_unified(model, test_loader)
    print(f"Epoch [{epoch}/{EPOCHS}] ({ep_time:.1f}s) - Loss: {avg_loss:.4f} | Disease Acc: {acc_d*100:.2f}% | Crop Acc: {acc_c*100:.2f}% | Consistency: {cons*100:.2f}% | Saliency: {mask_m:.3f}")
    
    history.append({
        "epoch": epoch,
        "loss": avg_loss,
        "disease_acc": acc_d,
        "crop_acc": acc_c,
        "consistency": cons,
        "saliency_mean": mask_m,
        "time_s": ep_time
    })
    
    if acc_d > best_acc:
        best_acc = acc_d
        torch.save({
            "model_state_dict": model.state_dict(),
            "best_acc": best_acc,
            "epoch": epoch
        }, save_path)
        print(f"  ⭐ New Best Unified Model Saved! Acc: {best_acc*100:.2f}%")

print(f"\nTraining Complete! Maximum Field Disease Accuracy: {best_acc*100:.2f}%")


In [ ]:
out_dir = Path("/kaggle/working/unified_cnn_results")
out_dir.mkdir(parents=True, exist_ok=True)

df_h = pd.DataFrame(history)
df_h.to_csv(out_dir / "unified_training_history.csv", index=False)

summary = {
    "initial_disease_acc": float(acc_d_0),
    "best_disease_acc": float(best_acc),
    "crop_family_acc": float(history[-1]["crop_acc"]),
    "cross_crop_consistency": float(history[-1]["consistency"]),
    "mean_saliency_gating": float(history[-1]["saliency_mean"])
}
with open(out_dir / "unified_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

import shutil
shutil.make_archive("/kaggle/working/unified_cnn_results", "zip", "/kaggle/working/unified_cnn_results")
print("✅ Unified Model Results Archived to /kaggle/working/unified_cnn_results.zip")
